# UCREL-NLP Summer School 2026

![header](https://r2.epsiotapi.com/Summer-School-2026/header_img.png)


# CoT

Chain-of-Thought (CoT) Prompting
==========================================================

Task: estimate body fat mass from a short word problem.
(unit conversion -> BMI -> body fat % -> body fat mass),
and each step can be silently skipped or miscalculated by an LLM that
just "jumps" to an answer.

Ground truth for the example question below:
    BMI                 = 24.8
    Body Fat Percentage = 19.78%
    Body Fat Mass        = 14.8 kg  (32.6 lbs)


##Install the OpenAI SDK and import the required Python libraries

In [1]:
!pip install openai

In [2]:
import os
from openai import OpenAI
from google.colab import userdata

In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_KEY')
)

# list the model provided by OpenRouter
models = client.models.list()
for model in models.data:
    print(model.id)

moonshotai/kimi-k3
meta/muse-spark-1.1
kwaipilot/kat-coder-air-v2.5
kwaipilot/kat-coder-pro-v2.5
openai/gpt-5.6-luna-pro
openai/gpt-5.6-luna
openai/gpt-5.6-terra-pro
openai/gpt-5.6-terra
openai/gpt-5.6-sol-pro
openai/gpt-5.6-sol
x-ai/grok-4.5
~x-ai/grok-latest
aion-labs/aion-3.0-mini
aion-labs/aion-3.0
tencent/hy3
tencent/hy3:free
poolside/laguna-xs-2.1
poolside/laguna-xs-2.1:free
anthropic/claude-sonnet-5
google/gemini-3.1-flash-lite-image
nex-agi/nex-n2-mini
sakana/fugu-ultra
google/gemini-3.1-flash-image
google/gemini-3-pro-image
cohere/north-mini-code:free
z-ai/glm-5.2
openrouter/fusion
moonshotai/kimi-k2.7-code
~anthropic/claude-fable-latest
anthropic/claude-fable-5
nex-agi/nex-n2-pro
nvidia/nemotron-3.5-content-safety:free
nvidia/nemotron-3-ultra-550b-a55b
nvidia/nemotron-3-ultra-550b-a55b:free
qwen/qwen3.7-plus
minimax/minimax-m3
stepfun/step-3.7-flash
anthropic/claude-opus-4.8-fast
anthropic/claude-opus-4.8
qwen/qwen3.7-max
x-ai/grok-build-0.1
google/gemini-3.5-flash
anthropic/

In [14]:
def get_api_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_KEY")
    except Exception:
        return os.environ.get("OPENROUTER_KEY")

API_KEY = get_api_key()
MODEL = "google/gemini-3.1-flash-lite"  # any solid instruct model works fine
#MODEL = "qwen/qwen3-next-80b-a3b-instruct:free"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY) if API_KEY else None


The formula used to estimate body fat percentage depends on the person's gender:

    *For males:
    Body Fat Percentage(%) = BMI×1.2 + 0.23×Age − 16.2

    *For females:
    Body Fat Percentage(%) = BMI×1.2 + 0.23×Age − 5.4

Finally, we can estimate the total body fat mass:

    Body Fat Mass = Body Fat Percentage × Weight




In [15]:
def ask(prompt: str, system_prompt: str) -> str:
    """Send one message and return just the text content."""
    if client is None:
        return "[No API key set — see FALLBACK_OUTPUTS below for pre-recorded results]"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=600,
        seed=10, # to use a fixed starting point
    )
    return response.choices[0].message.content



# 2. Task setup


In [16]:
TASK = """
Calculate the person's body fat mass.
Body Fat Mass = Body Fat Percentage x Body Weight
Body Fat Percentage (male)   = BMI x 1.2 + 0.23 x Age - 16.2
Body Fat Percentage (female) = BMI x 1.2 + 0.23 x Age - 5.4
BMI = Weight (kg) / Height (m)^2
"""

QUESTION = "My cousin is 7 ft tall, 27 years old and weighs 83 kg. Can you calculate the body fat mass in pounds?"

CORRECT_ANSWER_KG_male = 21.75
CORRECT_ANSWER_KG_female = 41.5175

# 3. Three prompting strategies

In [17]:
NO_COT = TASK + "\n Give only the final numeric answer in kg. Do not show your process."


ZERO_SHOT_COT = TASK + "\n Let's think step by step."


FEW_SHOT_COT = TASK + """
Examples:
Q: Woman, 46, 165 cm, 55 kg -> body fat mass?
A: BMI=55/1.65^2=20.2 | Fat%=20.2*1.2+0.23*46-5.4=29.4% | Mass=29.4%*55=16.2 kg -> 16.2 kg

Q2: man, 46, 5 ft, 55 kg -> body fat mass?
A2: BMI=55/((5*0.3048))^2=20.2 | Fat%=20.2*1.2+0.23*46-16.2=18.6% | Mass=18.6%*55=16.2 kg -> 10.23 kg

Let's think step by step. These are two examples to learn the exact procedure, showing each step, then give the final answer in kg.
Now solve the new question the same way with 4 decimal division.
Gender matters as you see in examples. If not sure, calculate for both.
"""

# 4. Run all three and compare


In [18]:


if __name__ == "__main__":
    strategies = {
        "No CoT": NO_COT,
        "Zero-shot CoT": ZERO_SHOT_COT,
        "Few-shot CoT": FEW_SHOT_COT,
    }

    print(f"Question: {QUESTION}")
    #print(f"Correct answer: ~{CORRECT_ANSWER_KG_male} kg\n")
    print("=" * 60)

    for name, system_prompt in strategies.items():
        answer = ask(QUESTION, system_prompt)
        print(f"\n--- {name} ---\n{answer}\n")
        print("=" * 60)

    print(
        "\nDiscussion: which strategy got closest to "
        f"{CORRECT_ANSWER_KG_male} kg, and why? Which steps did the 'No CoT' "
        "answer likely skip?"
    )

Question: My cousin is 7 ft tall, 27 years old and weighs 83 kg. Can you calculate the body fat mass in pounds?

--- No CoT ---
10.55


--- Zero-shot CoT ---
To calculate your cousin's body fat mass, we will follow these steps:

### Step 1: Convert measurements to metric
*   **Height:** 7 feet = 84 inches. 84 inches × 0.0254 = **2.1336 meters**.
*   **Weight:** **83 kg**.

### Step 2: Calculate BMI
*   BMI = Weight (kg) / Height (m)²
*   BMI = 83 / (2.1336)²
*   BMI = 83 / 4.5522
*   **BMI ≈ 18.23**

### Step 3: Calculate Body Fat Percentage
*Assuming your cousin is male (if female, the constant at the end changes from 16.2 to 5.4):*
*   Body Fat % = (BMI × 1.2) + (0.23 × Age) - 16.2
*   Body Fat % = (18.23 × 1.2) + (0.23 × 27) - 16.2
*   Body Fat % = 21.876 + 6.21 - 16.2
*   **Body Fat % ≈ 11.886%** (or 0.11886)

### Step 4: Calculate Body Fat Mass in kg
*   Body Fat Mass = Body Fat % × Body Weight
*   Body Fat Mass = 0.11886 × 83 kg
*   **Body Fat Mass ≈ 9.865 kg**

### Step 5: Conve

Use the following alternatives depending on your specific needs:

For general logical breakdown:
"Walk me through your reasoning process."
"Explain your reasoning."
"Break this down into logical parts."

For mathematical or calculation tasks:
"Show your step-by-step calculation."
"Work this out in a step-by-step way to be sure."

For research and analysis:
"First, think about this logically."
"List the pros and cons before concluding."
"Identify the constraints and work through them."

For formatting a numbered chain:
"Deconstruct this into numbered, sequential steps."
"Provide a step-by-step breakdown."

In [ ]:
import requests

response = requests.get("https://openrouter.ai/api/v1/models")
models = response.json()["data"]

free_models = [m["id"] for m in models if m["id"].endswith(":free")]
print(free_models)

['tencent/hy3:free', 'poolside/laguna-xs-2.1:free', 'cohere/north-mini-code:free', 'nvidia/nemotron-3.5-content-safety:free', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'poolside/laguna-m.1:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'nvidia/nemotron-3-super-120b-a12b:free', 'nvidia/nemotron-3-nano-30b-a3b:free', 'nvidia/nemotron-nano-12b-v2-vl:free', 'qwen/qwen3-next-80b-a3b-instruct:free', 'nvidia/nemotron-nano-9b-v2:free', 'openai/gpt-oss-20b:free', 'qwen/qwen3-coder:free', 'cognitivecomputations/dolphin-mistral-24b-venice-edition:free', 'meta-llama/llama-3.3-70b-instruct:free', 'meta-llama/llama-3.2-3b-instruct:free', 'nousresearch/hermes-3-llama-3.1-405b:free']


In [ ]:
import time
from openai import OpenAI, RateLimitError

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

def ask_with_retry(prompt, model, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            wait = 30  # OpenRouter often tells you this in retry_after_seconds
            print(f"Rate limited, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait)
    raise Exception("Max retries exceeded")